In [ ]:
#DAILY CHALLENGE

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(r'C:\Users\kmeso\Desktop\TTA_KOUAME_ASSE_SOKRY_JOSEPH\WEEK03\GLOBAL\global_power_plant_database.csv')

print(f"Dimensions : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"Colonnes : {list(df.columns)[:10]}...")

missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Manquantes': missing, 'Pourcentage (%)': missing_pct})
missing_df = missing_df[missing_df['Manquantes'] > 0].sort_values('Manquantes', ascending=False)

print("\nValeurs manquantes par colonne :")
print(missing_df.head(10))

cols_numeriques = ['capacity_mw', 'latitude', 'longitude', 'year_of_capacity_data']
for col in cols_numeriques:
    if col in df.columns:
        mediane = df[col].median()
        df[col] = df[col].fillna(mediane)

df = df.dropna(subset=['country', 'primary_fuel'])
print(f"\nAprès nettoyage : {df.shape[0]} lignes")

capacite_np = df['capacity_mw'].to_numpy(dtype=np.float64)
df['capacity_mw'] = capacite_np

print("\nStatistiques des colonnes numériques :")
print(df[['capacity_mw', 'latitude', 'longitude']].describe())

top_pays = df['country'].value_counts().head(10)
print("\nTop 10 pays par nombre de centrales :")
for i, (pays, count) in enumerate(top_pays.items(), 1):
    print(f"   {i}. {pays} : {count} centrales")

top_fuels = df['primary_fuel'].value_counts()
print("\nTop 10 types de combustible :")
for i, (fuel, count) in enumerate(top_fuels.head(10).items(), 1):
    print(f"   {i}. {fuel} : {count} centrales")

fuels = df['primary_fuel'].unique()[:5]  
stats_fuels = {}

for fuel in fuels:
    capacites = df[df['primary_fuel'] == fuel]['capacity_mw'].dropna().to_numpy()
    if len(capacites) > 0:
        stats_fuels[fuel] = {
            'count': len(capacites),
            'mean': np.mean(capacites),
            'median': np.median(capacites),
            'std': np.std(capacites),
            'min': np.min(capacites),
            'max': np.max(capacites),
            'q25': np.percentile(capacites, 25),
            'q75': np.percentile(capacites, 75)
        }

stats_df = pd.DataFrame(stats_fuels).T.round(2)
print(stats_df)

print("\nTEST D'HYPOTHÈSE : Puissance moyenne (Hydro vs Gaz)")
hydro_cap = df[df['primary_fuel'] == 'Hydro']['capacity_mw'].dropna().to_numpy()
gas_cap = df[df['primary_fuel'] == 'Gas']['capacity_mw'].dropna().to_numpy()

t_stat = (np.mean(hydro_cap) - np.mean(gas_cap)) / np.sqrt(
    np.var(hydro_cap)/len(hydro_cap) + np.var(gas_cap)/len(gas_cap)
)
print(f"t-statistique : {t_stat:.3f}")
if abs(t_stat) > 2:
    print("Différence significative entre Hydro et Gas (p < 0.05 approx.)")
else:
    print("Différence non significative")

df['year'] = pd.to_numeric(df['year_of_capacity_data'], errors='coerce')
df = df.dropna(subset=['year'])
df['year'] = df['year'].astype(int)

df_time = df[df['year'] >= 1970]
years = np.sort(df_time['year'].unique())

fuels_time = ['Coal', 'Gas', 'Hydro', 'Solar', 'Wind']
evolution = {}

for fuel in fuels_time:
    fuel_data = df_time[df_time['primary_fuel'] == fuel]
    count_by_year = fuel_data.groupby('year').size()
    evolution[fuel] = count_by_year.reindex(years, fill_value=0).to_numpy()

total_by_year = df_time.groupby('year').size().reindex(years, fill_value=0).to_numpy()
evolution_pct = {fuel: (evolution[fuel] / total_by_year) * 100 for fuel in fuels_time}

plt.figure(figsize=(12, 6))
for fuel in fuels_time:
    plt.plot(years, evolution_pct[fuel], linewidth=2, label=fuel)

plt.title('Évolution de la part des types de combustible (en nombre de centrales)', fontweight='bold')
plt.xlabel('Année')
plt.ylabel('Pourcentage des nouvelles centrales (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Visualisation : augmentation des énergies renouvelables (Solaire, Éolien) depuis 2000")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['capacity_mw'].dropna(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution des capacités des centrales (MW)')
axes[0].set_xlabel('Capacité (MW)')
axes[0].set_ylabel('Nombre de centrales')
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)

top5_fuels = df['primary_fuel'].value_counts().head(5).index
df_top5 = df[df['primary_fuel'].isin(top5_fuels)]

sns.boxplot(data=df_top5, x='primary_fuel', y='capacity_mw', ax=axes[1], palette='Set2')
axes[1].set_title('Capacité par type de combustible (top 5)')
axes[1].set_xlabel('Combustible')
axes[1].set_ylabel('Capacité (MW)')
axes[1].set_yscale('log')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("\nCarte des centrales (échantillon de 5000 points)")
sample = df.sample(min(5000, len(df)))
plt.figure(figsize=(12, 8))
sc = plt.scatter(sample['longitude'], sample['latitude'], 
                 c=sample['capacity_mw'], cmap='viridis', 
                 s=10, alpha=0.6, edgecolors='none')
plt.colorbar(sc, label='Capacité (MW)')
plt.title('Localisation géographique des centrales électriques')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

cols_corr = ['capacity_mw', 'latitude', 'longitude']
mat_corr = df[cols_corr].dropna().to_numpy()
corr_matrix = np.corrcoef(mat_corr.T)

print("Matrice de corrélation (NumPy) :")
print(np.round(corr_matrix, 3))

eigenvalues, eigenvectors = np.linalg.eig(corr_matrix)
print("\nValeurs propres :")
print(np.round(eigenvalues, 3))
print("\nVecteurs propres :")
print(np.round(eigenvectors, 3))

print("\nInterprétation :")
print("   • Les valeurs propres mesurent l'inertie (variance expliquée)")
print("   • Le premier vecteur propre indique la direction de plus grande variance")
print("   • Utile pour réduire la dimensionnalité (ACP)")

capacite_np = df['capacity_mw'].to_numpy()
latitude_np = df['latitude'].to_numpy()
mediane_cap = np.median(capacite_np)
mask = (capacite_np > mediane_cap) & (latitude_np > 0)
df_filtre = df[mask]

print(f"Filtrage avancé (capacité > {mediane_cap:.1f} MW et hémisphère nord)")
print(f"   {len(df_filtre)} centrales sélectionnées sur {len(df)}")

x = np.linspace(0, 10, 100)
y_base = df_filtre.groupby('primary_fuel').size().values[:5]
y_norm = y_base / np.max(y_base)

plt.figure(figsize=(10, 5))
plt.bar(df_filtre['primary_fuel'].value_counts().head(5).index, 
        df_filtre['primary_fuel'].value_counts().head(5).values,
        color=plt.cm.viridis(np.linspace(0, 0.8, 5)))
plt.title('Top 5 combustibles après filtrage (capacité > médiane, hémisphère nord)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()